# 실습 ③ 다중 벡터 검색기 + 최종 체인 — 연습본

> **[권장 난이도: 중급]** · **[추천 소요 시간: 50분]** · **[교육 과정: 12~13주차 · 7교시]**

색인 카드(요약문)를 **서랍**에 꽂고, 원본을 **서가**에 보관해 연결합니다. 오늘 수업의 마지막 조각입니다.

---

## 오늘 만드는 것

```
질문 ──▶ 요약으로 검색 ──▶ 원본 꺼내기 ──▶ 멀티모달 LLM ──▶ 답변
         (색인 카드)       (진짜 책)        (눈 달린 모델)
```

## 진행 방법

1. 셀을 **위에서부터 순서대로** 실행합니다 (`Shift + Enter`)
2. `# TODO` 와 `____` 가 있는 곳을 직접 채웁니다
3. 각 단계의 **확인 포인트**를 통과하면 다음으로 넘어갑니다

## ⚠️ 마지막 8번 셀을 꼭 하세요

**Before/After 비교**는 과제 채점에서 7점이고, 포트폴리오에서 가장 강력한 자료입니다. 시간이 모자라면 9번(번역 체인)을 건너뛰고 8번을 하세요.

## 0. 4~6교시 결과 복원

지금까지 만든 모든 재료를 한 번에 불러옵니다.

> ⚠️ `FileNotFoundError`가 나면 앞 교시의 저장 셀을 실행하지 않은 것입니다.

In [ ]:
import os
import re
import time
import uuid
import pickle
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OpenAI API 키가 없습니다. .env 파일을 확인하세요.")

processed = PROJECT_ROOT / "data" / "processed"

# 4교시 결과
with open(processed / "extracted.pkl", "rb") as f:
    d1 = pickle.load(f)
texts, tables, texts_2k_token = d1["texts"], d1["tables"], d1["texts_2k_token"]

# 5~6교시 결과
with open(processed / "summaries.pkl", "rb") as f:
    d2 = pickle.load(f)
text_summaries = d2["text_summaries"]
table_summaries = d2["table_summaries"]
image_summaries = d2["image_summaries"]
img_base64_list = d2["img_base64_list"]

print("복원 완료")
print(f"  텍스트  원본 {len(texts_2k_token):3d}건 / 요약 {len(text_summaries):3d}건")
print(f"  표      원본 {len(tables):3d}건 / 요약 {len(table_summaries):3d}건")
print(f"  이미지  원본 {len(img_base64_list):3d}건 / 요약 {len(image_summaries):3d}건")

> ✅ **확인 포인트:** 텍스트·표·이미지의 원본 수와 요약 수가 각각 같아야 합니다. (표는 0건일 수 있습니다)

## 1. 다중 벡터 검색기의 구조

> **비유:** **도서관의 색인 카드**입니다. 사서는 책을 다 읽고 찾지 않습니다. 요약이 적힌 **색인 카드로 검색**하고, 찾으면 **서가에서 진짜 책을 꺼내** 건네줍니다.

| 구성 요소 | 비유 | 들어가는 것 | 역할 |
| --- | --- | --- | --- |
| `vectorstore` (Chroma) | 색인 카드 서랍 | 요약문 임베딩 | **검색** |
| `docstore` (InMemoryStore) | 서가 | 원본 텍스트·표·Base64 이미지 | **전달** |
| `id_key` | 청구기호 | `doc_id` | 둘을 **연결** |

오늘 배우는 것의 핵심은 한 줄입니다.

> **요약과 원본에 같은 번호표를 붙인다.**

## 2. 검색기 만들기

아래 셀은 그대로 실행하세요.

> `InMemoryStore`는 메모리에만 저장됩니다. 커널을 재시작하면 사라지므로 실제 서비스에서는 파일이나 Redis를 씁니다.

In [ ]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

vectorstore = Chroma(
    collection_name="multimodal_rag",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
)

store = InMemoryStore()
id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)

print("검색기 생성 완료")

## 3. 요약과 원본을 같은 ID로 짝짓기

**TODO 1~2.** 빈칸 2개를 채우세요.

💡 **힌트:** 위 표를 보세요.
- 요약문은 **검색**에 쓰입니다 → 어디에 넣어야 할까요?
- 원본은 **전달**에 쓰입니다 → 어디에 넣어야 할까요?

> ⚠️ **이 셀을 두 번 실행하면 데이터가 중복됩니다.** 다시 돌리려면 2번 셀부터 재실행하세요.

In [ ]:
from langchain_core.documents import Document as _Doc


def add_documents(retriever, doc_summaries, doc_contents):
    """요약본은 벡터 DB에, 원본은 docstore에 '같은 id'로 저장합니다."""
    doc_ids = [str(uuid.uuid4()) for _ in doc_contents]

    summary_docs = [
        _Doc(page_content=s, metadata={id_key: doc_ids[i]})
        for i, s in enumerate(doc_summaries)
    ]

    # TODO 1: 요약본은 어디에 넣을까요? (vectorstore / docstore)
    retriever.____.add_documents(summary_docs)

    # TODO 2: 원본은 어디에 넣을까요?
    retriever.____.mset(list(zip(doc_ids, doc_contents)))


if text_summaries:
    add_documents(retriever, text_summaries, texts_2k_token)
if table_summaries:
    add_documents(retriever, table_summaries, tables)
if image_summaries:
    add_documents(retriever, image_summaries, img_base64_list)

print("색인 완료")
print("  벡터 DB에 저장된 요약 수:", vectorstore._collection.count())

## 4. 검색 테스트 — 요약으로 찾고 원본을 받는다

**TODO 3.** 본인 PDF에 맞는 질문을 적으세요.

💡 **힌트:** **그래프나 표를 봐야만** 답할 수 있는 질문이어야 합니다. 텍스트에도 똑같이 적혀 있는 내용을 물으면 나중에 Before/After 차이가 안 납니다.

> 좋은 예: "그래프에 나타난 2023년 추세를 설명해줘"
> 나쁜 예: "이 문서는 무엇에 관한 것인가요?"

In [ ]:
# TODO 3: 그래프나 표를 봐야만 답할 수 있는 질문을 적으세요
query = "____"

found = retriever.invoke(query)

print("검색된 원본 수:", len(found))
print()

for i, doc in enumerate(found):
    content = doc.page_content if hasattr(doc, "page_content") else doc
    is_image = bool(re.match("^[A-Za-z0-9+/]{100,}={0,2}$", content))
    kind = "이미지(Base64)" if is_image else "텍스트/표"
    print(f"[{i}] {kind} — 길이 {len(content):,}자")
    print("   ", content[:120].replace("\n", " "), "...")
    print()

> ✅ **확인 포인트:** 검색 결과에 **이미지(Base64)가 최소 1건** 들어 있어야 합니다.
>
> 하나도 없다면 이미지 요약의 품질이 낮아 검색이 안 되는 것입니다. 6교시로 돌아가 프롬프트를 보강하세요.

> 💡 **여기가 오늘의 핵심 장면입니다.** 검색어와 글자가 하나도 안 겹치는 원본이 나왔죠? 요약문을 거쳐서 찾았기 때문입니다.

## 5. 검색 결과를 텍스트와 이미지로 분리

멀티모달 LLM에 넘기려면 둘을 다른 형식으로 포장해야 합니다. Base64 문자열의 **생김새**로 구분합니다.

아래 셀은 그대로 실행하세요.

In [ ]:
from langchain_core.documents import Document


def split_image_text_types(docs):
    """검색 결과를 Base64 이미지와 텍스트로 나눕니다."""
    b64_images = []
    texts_out = []

    for doc in docs:
        doc = doc.page_content if isinstance(doc, Document) else doc
        # Base64 문자열은 영문/숫자/+/= 만으로 길게 이어집니다
        if re.match("^[A-Za-z0-9+/]{100,}={0,2}$", doc):
            b64_images.append(doc)
        else:
            texts_out.append(doc)

    return {"images": b64_images, "texts": texts_out}


# 앞서 검색한 결과로 동작 확인
split = split_image_text_types(found)
print("이미지:", len(split["images"]), "건")
print("텍스트:", len(split["texts"]), "건")

## 6. 최종 멀티모달 체인 만들기

체인의 구조는 세 칸입니다.

```
{검색 + 질문}  →  img_prompt_func  →  멀티모달 LLM  →  StrOutputParser
   재료 모으기      메시지로 조립        답변 생성        문자열로 꺼내기
```

**TODO 4~5.** 빈칸 2개를 채우세요.

💡 **힌트**
- TODO 4: 6교시에서 Base64 앞에 붙였던 것과 같은 접두사입니다
- TODO 5: 위 그림의 3번 칸에 들어갈 변수 이름입니다

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI


def img_prompt_func(data_dict):
    """검색된 원본을 멀티모달 LLM이 읽을 형태로 조립합니다."""
    messages = []

    # 1) 이미지를 먼저 넣습니다 (모델이 그림을 먼저 보게)
    for image in data_dict["context"]["images"]:
        messages.append({
            "type": "image_url",
            # TODO 4: Base64 앞에 붙일 접두사를 완성하세요
            "image_url": {"url": f"____{image}"},
        })

    # 2) 텍스트 컨텍스트와 질문을 넣습니다
    formatted_texts = "\n".join(data_dict["context"]["texts"])
    messages.append({
        "type": "text",
        "text": (
            "아래 자료를 근거로 질문에 한국어로 답하세요.\n"
            "자료에 없는 내용은 모른다고 답하세요.\n\n"
            f"질문: {data_dict['question']}\n\n"
            f"참고 텍스트:\n{formatted_texts}"
        ),
    })

    return [HumanMessage(content=messages)]


model = ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)

chain = (
    {
        "context": retriever | RunnableLambda(split_image_text_types),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(img_prompt_func)
    | ____                     # TODO 5: 답변을 생성할 모델
    | StrOutputParser()
)

print("최종 체인 조립 완료")

## 7. 질의 실행

In [ ]:
start = time.time()
answer = chain.invoke(query)

print(f"답변 생성 — {time.time() - start:.1f}초\n")
print("Q:", query)
print("-" * 70)
print(answer)

> ✅ **확인 포인트:** 답변에 **그래프나 표에서만 알 수 있는 내용**이 들어 있어야 합니다.
>
> 두루뭉술한 답이 나왔다면 질문을 더 구체적으로 바꿔보세요.

## 8. Before / After 비교 (가장 중요)

> 📌 **과제 채점 7점짜리 셀입니다. 반드시 하세요.**

같은 질문을 **텍스트 전용 RAG**와 **멀티모달 RAG**에 각각 던져 차이를 증거로 남깁니다.

먼저 비교용 텍스트 전용 RAG를 만듭니다. **이미지와 표를 뺀** 것 말고는 모든 조건이 같습니다.

In [ ]:
# 텍스트만 들어간 벡터 DB (= 11주차에 만든 것과 같은 구조)
text_only_store = Chroma(
    collection_name="text_only_rag",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
)
text_only_store.add_texts(texts_2k_token)
text_only_retriever = text_only_store.as_retriever(search_kwargs={"k": 4})

text_only_prompt = ChatPromptTemplate.from_template(
    "아래 자료를 근거로 질문에 한국어로 답하세요.\n"
    "자료에 없는 내용은 모른다고 답하세요.\n\n"
    "질문: {question}\n\n참고 텍스트:\n{context}"
)

text_only_chain = (
    {
        "context": text_only_retriever | (lambda docs: "\n".join(d.page_content for d in docs)),
        "question": RunnablePassthrough(),
    }
    | text_only_prompt
    | model
    | StrOutputParser()
)

print("비교용 텍스트 전용 RAG 준비 완료")

In [ ]:
print("=" * 70)
print("질문:", query)
print("=" * 70)
print()

print("[BEFORE] 텍스트 전용 RAG")
print("-" * 70)
before = text_only_chain.invoke(query)
print(before)
print()

print("[AFTER] 멀티모달 RAG")
print("-" * 70)
after = chain.invoke(query)
print(after)
print()

print("=" * 70)
print("이 출력 전체를 캡처해 README.md에 넣으세요.")
print("=" * 70)

**TODO 6. 두 답변의 차이를 적으세요 (더블클릭해서 편집)**

- 텍스트 전용 RAG의 답변은 어땠나요:
- 멀티모달 RAG의 답변은 어땠나요:
- 차이가 생긴 이유는:

> 💡 **차이가 잘 안 드러난다면** 질문이 잘못된 것입니다. 텍스트에도 같은 내용이 적혀 있을 가능성이 큽니다. **그래프에만 있는 정보**를 묻는 질문으로 바꿔 4번 셀부터 다시 돌리세요.

> 📸 **지금 이 출력을 스크린샷으로 저장하세요.** 포트폴리오에서 가장 강력한 자료입니다.

## 9. 번역 체인 통합 (선택)

Ollama가 설치된 수강생만 실행합니다. 번역만 로컬 모델에 맡겨 비용을 줄이는 구성입니다.

> ⏱️ 시간이 모자라면 건너뛰고 10번으로 가세요.

In [ ]:
# Ollama + llama3.1:8b 가 떠 있는 경우에만 주석을 해제하세요
#
# from langchain_ollama.llms import OllamaLLM
#
# translate_prompt = ChatPromptTemplate.from_template(
#     "다음 영문 답변을 자연스러운 한국어로 번역하세요. "
#     "숫자와 고유명사는 그대로 두세요.\n\n{answer}"
# )
# translator = OllamaLLM(model="llama3.1:8b")
#
# final_chain = (
#     chain
#     | (lambda a: {"answer": a})
#     | translate_prompt
#     | translator
#     | StrOutputParser()
# )
# print(final_chain.invoke(query))

print("번역 체인은 주석 해제 후 실행하세요 (Ollama 필요)")

## 10. 질의응답 5건 기록 (과제 제출물)

과제 제출물 6번이 "질의응답 5건"입니다. 이 셀이 README에 붙일 표까지 만들어 줍니다.

> ⏱️ 5건 모두 API를 호출하므로 1~2분 걸립니다.

In [ ]:
# 과제 제출물 6번 "질의응답 5건"을 그대로 만들어 주는 셀입니다.
questions = [
    query,
    "이 문서의 핵심 주제를 한 문장으로 요약해줘",
    "문서에 나온 표의 내용을 설명해줘",
    "가장 중요한 수치 3가지를 알려줘",
    "이 문서가 다루지 않는 내용은 무엇인지 알려줘",
]

rows = []
for q in questions:
    found_q = retriever.invoke(q)
    kinds = split_image_text_types(found_q)
    answer = chain.invoke(q)
    rows.append({
        "질문": q,
        "근거": f"이미지 {len(kinds['images'])}건 / 텍스트 {len(kinds['texts'])}건",
        "답변": answer,
    })
    print("=" * 70)
    print("Q:", q)
    print("근거:", rows[-1]["근거"])
    print("A:", answer[:400])
    print()

# README에 붙일 마크다운 표로 출력
print("\n" + "=" * 70)
print("아래를 복사해 README.md에 붙여넣으세요")
print("=" * 70 + "\n")
print("| 질문 | 근거가 된 원본 | 답변 |")
print("| --- | --- | --- |")
for r in rows:
    short = r["답변"].replace("\n", " ")[:120]
    print(f"| {r['질문']} | {r['근거']} | {short}... |")

---

## ✅ 체크포인트 4 — 모두 체크되면 7교시 완료

- [ ] 질문 하나에 대해 답변이 출력된다
- [ ] 답변에 **그래프나 표에서만 알 수 있는 내용**이 들어 있다
- [ ] 검색 결과에 이미지(Base64)가 최소 1건 포함된다
- [ ] Before/After 출력을 **캡처해 저장**했다
- [ ] 질의응답 5건 마크다운 표를 복사해 두었다
- [ ] TODO 1~6을 모두 채웠다

## 생각해 볼 질문 (답을 적어보세요)

**Q1. 요약과 원본에 같은 `doc_id`를 붙이지 않으면 어떤 일이 벌어질까요?**

답:

**Q2. 검색은 요약으로 하는데, 왜 답변 생성에는 원본을 쓸까요? 요약만 쓰면 안 되나요?**

답:

**Q3. `InMemoryStore`는 커널을 끄면 사라집니다. 실제 서비스로 만들려면 무엇을 바꿔야 할까요?**

답:

## 자주 나는 에러

| 증상 | 원인 | 해결 |
| --- | --- | --- |
| `FileNotFoundError: summaries.pkl` | 5~6교시 저장 셀 미실행 | 앞 노트북 마지막 셀 재실행 |
| 검색 결과에 이미지가 없음 | 이미지 요약 품질이 낮음 | 6교시로 돌아가 프롬프트 보강 |
| 답변이 "자료에 없습니다" | 검색이 엉뚱한 원본을 가져옴 | 질문을 요약문 표현에 가깝게 수정 |
| 같은 원본이 여러 번 나옴 | `add_documents`를 두 번 실행 | 2번 셀부터 재실행 |
| `RateLimitError` | 질의 5건 연속 호출 | 잠시 대기 후 재실행 |
| Before/After 차이가 없음 | 질문이 텍스트로도 답 가능 | 그래프에만 있는 정보를 묻도록 수정 |
| `NameError: found` | 4번 셀을 안 돌림 | 위에서부터 순서대로 재실행 |

## 과제로 가져갈 것

| 산출물 | 어디서 나왔나 |
| --- | --- |
| Before/After 캡처 | 8번 셀 출력 |
| 질의응답 5건 표 | 10번 셀이 출력한 마크다운 |
| 아키텍처 다이어그램 | 직접 작도 (Excalidraw / Mermaid) |
| `README.md` | 문제 정의 + 위 자료들 |
| `.gitignore` | `.env`, `data/figures/`, `*.pdf` |

> ⚠️ 마지막으로 반드시 확인: **`.env`가 `.gitignore`에 들어 있는지.** 이것 하나로 과제 10점이 갈립니다.